In [ ]:
import random
import sys
import numpy as np
import torch

from functools import partial
from pathlib import Path
from pycocotools.coco import COCO
from torch.utils.data import DataLoader

PROJECT_ROOT = Path("/kaggle/working")
INPUT_ROOT = Path("/kaggle/input")
COMPETITION = "multiple-object-video-detection-hw-2-ucu-ai-2026"

DIRECTORY = INPUT_ROOT / COMPETITION
if not (DIRECTORY / "train" / "annotations_coco.json").is_file():
    data_roots = sorted({path.parent.parent
                         for path in INPUT_ROOT.rglob("annotations_coco.json")
                         if path.parent.name == "train"})
    if len(data_roots) != 1:
        raise FileNotFoundError(
            "Attach the extracted competition data, or set DIRECTORY to its input folder."
        )
    DIRECTORY = data_roots[0]

INFO_TRAIN = DIRECTORY / "train" / "annotations_coco.json"
INFO_VAL = DIRECTORY / "val" / "annotations_coco.json"
INFO_TEST = DIRECTORY / "test_images.csv"

required_scripts = {
    "data_prep.py",
    "data_cv.py",
    "dataset_helpers.py",
    "torch_model.py",
    "yolo26_model.py",
    "experiment_runner.py",
    "experiment_results.py",
}

script_dirs = sorted({path.parent for path in INPUT_ROOT.rglob("torch_model.py")
    if path.parent.name == "scripts" and all((path.parent / filename).is_file() for filename in required_scripts)
})
scripts_parent = str(script_dirs[0].parent)

if scripts_parent not in sys.path:
    sys.path.insert(0, scripts_parent)

from scripts.data_prep import load_annotations
from scripts.data_cv import stratified_group_folds
from scripts.dataset_helpers import DetectionDataset, collate_data
from scripts.experiment_runner import run_torch_cv
from scripts.experiment_results import model_training_results

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

SEED = 42
FOLDS = 3
EPOCHS = 5
FRAME_STEP = 15
BATCH_SIZE = 2

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

In [ ]:
train_frames, train_boxes = load_annotations(COCO(str(INFO_TRAIN)), split="train")
frames = stratified_group_folds(train_frames, train_boxes, n_splits=FOLDS, seed=SEED)
annotations_by_image = {image_id: group for image_id, group in train_boxes.groupby("image_id")}
empty_annotations = train_boxes.iloc[:0].copy()

loading annotations into memory...
Done (t=6.75s)
creating index...
index created!


In [ ]:
cv_fold_loaders = {}

kwargs_dataset = {
    "directory": DIRECTORY,
    "annotations_by_image": annotations_by_image,
    "empty_annotations": empty_annotations,
}

for fold in range(FOLDS):
    training_frames = (frames.loc[frames[f"train_fold_{fold}"].eq(True)].sort_values(["video", "frame_id"])
        .groupby("video", group_keys=False).nth[::FRAME_STEP].reset_index(drop=True))

    validation_frames = frames.loc[frames["validation_fold"].eq(fold)].reset_index(drop=True)

    train_dataset = DetectionDataset(frames=training_frames, validation=False, **kwargs_dataset)
    valid_dataset = DetectionDataset(frames=validation_frames, validation=True, **kwargs_dataset)

    cv_fold_loaders[fold] = {
        "train": DataLoader(train_dataset,
            batch_size=BATCH_SIZE,
            shuffle=True,
            num_workers=0,
            collate_fn=collate_data
        ),
        "valid": DataLoader(
            valid_dataset,
            batch_size=BATCH_SIZE,
            shuffle=False,
            num_workers=0,
            collate_fn=collate_data
        )
    }

    print(f"Fold {fold}: {len(train_dataset)} training frames, {len(valid_dataset)} validation frames")

Fold 0: 1107 training frames, 7943 validation frames
Fold 1: 1094 training frames, 8088 validation frames
Fold 2: 1091 training frames, 8170 validation frames


## ROI

In [ ]:
ROI_histories, ROI_results = run_torch_cv(**cv_kwargs,
                                          model_method=partial(build_frcnn_model, roi_positive_fraction=0.5),
                                          output_dir=PROJECT_ROOT / "outputs" / "frcnn_roi",
                                          lr_scheduler=False)

In [ ]:
ROI_results_wrap = model_training_results(ROOT=PROJECT_ROOT, filename="frcnn_roi", cv_folds=range(FOLDS))


Fold 0


,epoch,train_loss,learning_rate,mAP,AP50,AP_small
0,1,1.866423,0.0001,0.093827,0.227035,0.030214
1,2,1.587907,0.0001,0.122384,0.260608,0.034855
2,3,1.476855,0.0001,0.123232,0.278409,0.048012
3,4,1.396038,0.0001,0.127860,0.288650,0.037923
4,5,1.325353,0.0001,0.148626,0.325543,0.054921



Fold 1


,epoch,train_loss,learning_rate,mAP,AP50,AP_small
0,1,1.916987,0.0001,0.116547,0.250920,0.039016
1,2,1.621475,0.0001,0.145235,0.311696,0.051212
2,3,1.501148,0.0001,0.131955,0.288151,0.042613
3,4,1.410724,0.0001,0.158130,0.330445,0.046166
4,5,1.321903,0.0001,0.161628,0.331951,0.052332



Fold 2


,epoch,train_loss,learning_rate,mAP,AP50,AP_small
0,1,1.803412,0.0001,0.088210,0.200394,0.038491
1,2,1.524521,0.0001,0.101430,0.226185,0.042188
2,3,1.405969,0.0001,0.100910,0.220280,0.042262
3,4,1.324079,0.0001,0.114827,0.243594,0.044557
4,5,1.264013,0.0001,0.119536,0.251405,0.049173



Best results per fold


,fold,epoch,train_loss,learning_rate,mAP,AP50,AP_small
0,0,5,1.325353,0.0001,0.148626,0.325543,0.054921
1,1,5,1.321903,0.0001,0.161628,0.331951,0.052332
2,2,5,1.264013,0.0001,0.119536,0.251405,0.049173



Metrics mean and std across folds


,mAP,AP50,AP_small
mean,0.143263,0.302966,0.052142
std,0.021552,0.044768,0.002879
